# Appendix B: Pingouin and Friends, a Cheat Sheet

This notebook runs every piece of code in the chapter, in order, with the chapter's text around it. Read the chapter on the site: [Appendix B: Pingouin and Friends, a Cheat Sheet](https://books.sugiv.fyi/lift/pingouin-cheatsheet.html). Everything is simulated from seeded code in `sim/`, so a run gives the same numbers as the book.

In [1]:
# Run this cell first. In Google Colab it fetches the companion repo; anywhere else, run the
# notebook from a clone of it. Either way it moves to the repo's root so `sim` can be imported.
import os
import pathlib
import sys

if "google.colab" in sys.modules:
    if not pathlib.Path("/content/lift-not-luck").exists():
        !git clone -q --depth 1 https://github.com/sugix/lift-not-luck /content/lift-not-luck
    %pip install -q pingouin
candidates = [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, pathlib.Path("/content/lift-not-luck")]
root = next(p for p in candidates if (p / "sim" / "parlo.py").is_file())
os.chdir(root)
sys.path.insert(0, str(root))

Every statistical call this book makes, on one page, with the output columns you'll actually read. This page runs on every build, so the examples and column names match the installed version of Pingouin.

In [2]:
import numpy as np
import pandas as pd
import pingouin as pg

print("pingouin", pg.__version__)
rng = np.random.default_rng(0)
control = rng.binomial(1, 0.38, 4_000)        # retained or not
treated = rng.binomial(1, 0.40, 4_000)

pingouin 0.7.0


## Comparing two groups

**`pg.ttest(x, y)`**: the difference in means between two groups. For a 0/1 metric like retention this is the usual A/B test. Read `T`, `p_val`, `CI95` (for the difference in means) and `cohen_d`. Ignore the `power` column: it's computed from the observed effect, which just restates the p-value (Chapter 1).

In [3]:
pg.ttest(treated, control)

,T,dof,alternative,p_val,CI95,cohen_d,power,BF10
T_test,2.685763,7998,two-sided,0.007251,"[0.01, 0.05]",0.060055,0.765922,0.921


**`pg.mwu(x, y)`**: Mann-Whitney U, a rank-based comparison for skewed metrics like revenue, when a few big values dominate the mean. It tests whether one group tends to have larger values, not whether the means differ.

In [4]:
revenue_c = rng.lognormal(1.0, 1.2, 2_000)
revenue_t = rng.lognormal(1.05, 1.2, 2_000)
pg.mwu(revenue_t, revenue_c)

,U_val,alternative,p_val,RBC,CLES
MWU,1972220.0,two-sided,0.44685,-0.01389,0.493055


## Planning a test

**`pg.power_ttest(d=..., n=..., power=...)`**: leave exactly one argument out and Pingouin solves for it. For a difference in rates, use $d = \delta / \sqrt{p(1-p)}$ (Chapter 2).

In [5]:
p, lift = 0.38, 0.015
d = lift / np.sqrt(p * (1 - p))
print(f"learners per arm for 80% power: {pg.power_ttest(d=d, power=0.8):,.0f}")
print(f"power with 10,000 per arm:       {pg.power_ttest(d=d, n=10_000):.0%}")

learners per arm for 80% power: 16,438
power with 10,000 per arm:       59%


**`pg.power_ttest2n(nx, ny, d=...)`**: the same for unequal arms, such as a 10% holdout (Chapter 10).

In [6]:
print(f"power, 45,000 vs 5,000: {pg.power_ttest2n(nx=45_000, ny=5_000, d=d):.0%}")

power, 45,000 vs 5,000: 55%


For a guardrail tested one-sided against a margin, pass `alternative="greater"` and use the gap between the margin and the expected harm as the effect (Chapter 2).

## Counts and proportions

**`pg.chi2_independence(df, x, y)`**: are two categorical columns related? It returns three things: expected counts, observed counts, and a stats table. Read the `pearson` row's `chi2` and `pval`. The book uses it to find *where* a sample ratio mismatch lives (Chapter 3) and to compare holdout and bandit groups (Chapter 10).

In [7]:
df = pd.DataFrame({"arm": rng.choice(["control", "treatment"], 6_000),
                   "platform": rng.choice(["iOS", "Android", "web"], 6_000)})
expected, observed, stats_table = pg.chi2_independence(df, x="platform", y="arm")
stats_table.loc[stats_table.test == "pearson", ["chi2", "dof", "pval"]]

,chi2,dof,pval
0,5.431501,2.0,0.066155


For an overall sample ratio check against an expected split, `scipy.stats.chisquare` is simpler:

In [8]:
from scipy import stats

print(stats.chisquare([15_733, 15_076]).pvalue)

0.00018179417226347284


## Uncertainty without formulas

**`pg.compute_bootci(x, y=None, func=..., paired=..., n_boot=...)`**: a bootstrap confidence interval for any statistic. With `paired=True`, pairs `(x[i], y[i])` are resampled together, which is how you keep each learner's numbers together for a ratio metric (Chapter 2).

In [9]:
lessons = rng.poisson(5, 2_000) + 1
done = rng.binomial(lessons, 0.78)
ci = pg.compute_bootci(lessons, done, func=lambda x, y: y.sum() / x.sum(),
                       paired=True, n_boot=2_000, seed=1)
print(f"completion per lesson, 95% CI {ci[0]:.3f} to {ci[1]:.3f}")

completion per lesson, 95% CI 0.780 to 0.790


## Adjusting for covariates

**`pg.ancova(data, dv, between, covar)`**: compare groups while adjusting for a covariate. With a pre-period covariate it's the same idea as CUPED (Chapter 5). Read the row for your group variable.

In [10]:
pre = rng.poisson(10, 6_000)
arm = rng.choice(["control", "treatment"], 6_000)
post = rng.poisson(pre * 0.8 + 2 + 0.3 * (arm == "treatment"))
pg.ancova(data=pd.DataFrame({"post": post, "pre": pre, "arm": arm}),
          dv="post", between="arm", covar="pre")

,Source,SS,DF,F,p_unc,np2
0,arm,131.640718,1,12.916797,0.000328,0.002149
1,pre,38978.820783,1,3824.664130,0.000000,0.389411
2,Residual,61118.043379,5997,NaN,NaN,NaN


**`pg.logistic_regression(X, y)`**: logistic regression with standard errors and confidence intervals. The book uses it to fit a surrogate index (Chapter 6). For the causal estimators in Part III, statsmodels' formula interface is more convenient (below).

In [11]:
X = pd.DataFrame({"active_days": rng.integers(0, 8, 5_000), "lessons": rng.poisson(5, 5_000)})
y = (rng.random(5_000) < 1 / (1 + np.exp(-(-2 + 0.35 * X.active_days + 0.05 * X.lessons)))).astype(int)
pg.logistic_regression(X, y)[["names", "coef", "se", "pval"]]

,names,coef,se,pval
0,Intercept,-2.040512,0.097548,3.671698e-97
1,active_days,0.370104,0.014701,7.575503e-140
2,lessons,0.037681,0.014058,7.352617e-03


## Many metrics

**`pg.multicomp(pvals, method="fdr_bh")`**: Benjamini-Hochberg correction. Returns a reject flag and adjusted p-values for each metric (Chapter 6). `method="bonf"` gives Bonferroni.

In [12]:
pvals = np.array([0.0001, 0.003, 0.02, 0.04, 0.2, 0.5, 0.8])
reject, adjusted = pg.multicomp(pvals, method="fdr_bh")
pd.DataFrame({"p": pvals, "BH adjusted": adjusted.round(3), "keep": reject})

,p,BH adjusted,keep
0,0.0001,0.001,True
1,0.0030,0.010,True
2,0.0200,0.047,True
3,0.0400,0.070,False
4,0.2000,0.280,False
5,0.5000,0.583,False
6,0.8000,0.800,False


## Beyond Pingouin

The book builds platform methods from scratch, because seeing them in ten lines of numpy is the best way to understand them. Where to find each:

| Method | Where | Chapter |
|---|---|---|
| Thompson sampling, UCB, epsilon-greedy | `sim/parlo.py` | 7 |
| Delta method for ratio metrics | `sim/design.py` (`delta_ratio`) | 2 |
| CUPED | `sim/faster.py` (`cuped`) | 5 |
| Always-valid p-values and confidence sequences | `sim/faster.py` (`msprt`, `confidence_sequence`) | 5 |
| Off-policy evaluation (IPS, SNIPS, doubly robust) | `sim/context.py` | 9 |
| IPW and doubly robust ATE, cross-fitted double ML | `sim/observe.py` | 11 |
| Synthetic control weights | `sim/quasi.py` (`synth_weights`) | 12 |
| Wald (IV) estimator | `sim/interfere.py` (`wald`) | 13 |
| T-learner, DR-learner, uplift curves, policy values | `sim/uplift.py` | 14 |
| A complete readout engine | `sim/engine.py` | 15 |

And the two libraries that carry Part III:

In [13]:
import statsmodels.formula.api as smf

frame = pd.DataFrame({"retained": y, "juno": rng.integers(0, 2, 5_000), **X})
model = smf.logit("retained ~ juno + active_days + lessons", frame).fit(disp=0)
effect = (model.predict(frame.assign(juno=1)) - model.predict(frame.assign(juno=0))).mean()
print(f"regression adjustment (g-formula) effect: {effect:+.4f}")

regression adjustment (g-formula) effect: +0.0011


- **statsmodels**: regressions with formulas, clustered standard errors (`cov_type="cluster"`, Chapter 12) and robust ones (`cov_type="HC1"`).
- **scikit-learn**: flexible models for propensities and outcomes in double ML and uplift models (Chapters 11 and 14). Use `KFold` for cross-fitting, so no learner's prediction comes from a model that saw them.